In [2]:
import json

In [7]:
from pydantic import BaseModel, Field

class Eval(BaseModel):
    completeness: float = Field(
        description="How complete is the answer in addressing all aspects of the question? 1 (very poor - missing key information) to 10 (ideal - all the information from the reference answer is provided completely). Only answer 10 if ALL information from the reference answer is included."
    )
    relevance: float = Field(
        description="How relevant is the answer to the specific question asked? 1 (very poor - off-topic) to 10 (ideal - directly addresses question and gives no additional information). Only answer 10 if the answer is completely relevant to the question and gives no additional information."
    )
    correctness: float = Field(
        description="How factually correct and accurate is the answer compared to the reference answer? 1 (very poor - contains major factual errors, hallucinations, or incorrect claims) to 10 (ideal - completely factually accurate with no incorrect or misleading information). An acceptable answer would score 6."
    )

In [6]:
from litellm import completion

In [14]:

def evaluate_answer(question, answer, model_response):
    """
    Evaluate answer quality using LLM-as-a-judge.
    """

    # LLM judge prompt
    judge_messages = [
        {
            "role": "system",
            "content": "You are an expert evaluator assessing the quality of answers. Evaluate the generated answer by comparing it to the reference answer. Only give 10/10 scores for perfect answers.",
        },
        {
            "role": "user",
            "content": f"""Question:
                {question}

                Generated Answer:
                {model_response}

                Reference Answer:
                {answer}

                Please evaluate the generated answer on three dimensions:
                1. Correctness: How factually correct is it compared to the reference answer? Only give 10/10 scores for perfect answers.
                2. Completeness: How thoroughly does it address all aspects of the question, covering all the information from the reference answer?
                3. Relevance: How well does it directly answer the specific question asked, giving no additional information?

                Provide scores from 1 (very poor) to 10 (ideal) for each dimension. If the answer is wrong, then the accuracy score must be 1.
            """,
        },
    ]

    # Call LLM judge with structured outputs (async)
    judge_response = completion(model="gpt-5.4-nano", messages=judge_messages, response_format=Eval)

    return Eval.model_validate_json(judge_response.choices[0].message.content)


Test response generation with finetuned Llama-3.1-8B-Instruct was possible only for the first 66 cases because of Google Colab error. So, for the sake of uniformity, we have considered only these test cases for evaluation of all the three techniques.

In [17]:
def evaluate_technique(file_name):

    completeness_score = 0.0
    relevance_score = 0.0
    correctness_score = 0.0

    with open(file_name, "r") as f:
        for _ in range(66):
            line = f.readline()
            line = json.loads(line)
            question = line["question"]
            answer = line["answer"]
            model_response = line["model_response"]
            result = evaluate_answer(question, answer, model_response)
            completeness_score += result.completeness
            relevance_score += result.relevance
            correctness_score += result.correctness

    completeness_score /= 66
    relevance_score /= 66
    correctness_score /= 66

    print(f"Avg Completeness score: {completeness_score:.2f}")
    print(f"Avg Relevance score: {relevance_score:.2f}")
    print(f"Avg Correctness score: {correctness_score:.2f}")


## RAG Evaluation

In [18]:
evaluate_technique("responses/rag_responses.jsonl")

/Users/ayanmukherjee/projects/llm_engineering/.venv/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='{"comple...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


Avg Completeness score: 7.06
Avg Relevance score: 8.56
Avg Correctness score: 7.30


## Open Source Fine Tuning Evaluation

In [19]:
evaluate_technique("responses/open_source_responses.jsonl")

Avg Completeness score: 3.15
Avg Relevance score: 5.95
Avg Correctness score: 2.48


## Closed Source Fine Tuning Evaluation

In [20]:
evaluate_technique("responses/closed_source_responses.jsonl")

Avg Completeness score: 3.23
Avg Relevance score: 4.80
Avg Correctness score: 2.86


# RAG is the winner !!!!!!